<a href="https://colab.research.google.com/github/Hamdaan07636/Hamdaan-Alam_2026B03110241/blob/main/Copy_of_Water_Consumption.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Project 25: Water Usage & Conservation Analysis (basic version)
import pandas as pd
import numpy as np

# Task 1: load the data and look at it
df = pd.read_csv("water_consumption_expanded.csv")
print("Task 1")
print(df.shape)
print(df.columns)
print(df.dtypes)



Task 1
(1250, 7)
Index(['Meter_Record_ID', 'Household_ID', 'Reading_Date', 'Household_Type',
       'Building_Type', 'Zone', 'Water_Usage_Liters'],
      dtype='object')
Meter_Record_ID        object
Household_ID           object
Reading_Date           object
Household_Type         object
Building_Type          object
Zone                   object
Water_Usage_Liters    float64
dtype: object


In [ ]:
# Task 2: missing values and duplicate meter IDs
print("\nTask 2")
print(df.isnull().sum())
print("Duplicate Meter_Record_ID:", df["Meter_Record_ID"].duplicated().sum())
df = df.drop_duplicates(subset="Meter_Record_ID")



Task 2
Meter_Record_ID        0
Household_ID           0
Reading_Date           0
Household_Type         0
Building_Type          0
Zone                   0
Water_Usage_Liters    30
dtype: int64
Duplicate Meter_Record_ID: 12


In [ ]:
# Task 3: fill missing usage with the median of each household type
# (median is not affected by very big or very small values)
print("\nTask 3")
medians = df.groupby("Household_Type")["Water_Usage_Liters"].transform("median")
df["Water_Usage_Liters"] = df["Water_Usage_Liters"].fillna(medians)
print("Missing now:", df["Water_Usage_Liters"].isnull().sum())



Task 3
Missing now: 0


In [ ]:
# Task 4: dates, month and weekday
print("\nTask 4")
df["Reading_Date"] = pd.to_datetime(df["Reading_Date"])
df["Month"] = df["Reading_Date"].dt.month_name()
df["Year_Month"] = df["Reading_Date"].dt.to_period("M")
df["Weekday"] = df["Reading_Date"].dt.day_name()
print(df[["Reading_Date", "Month", "Weekday"]].head())



Task 4
  Reading_Date     Month   Weekday
0   2025-11-16  November    Sunday
1   2025-08-14    August  Thursday
2   2026-06-30      June   Tuesday
3   2026-01-10   January  Saturday
4   2025-03-07     March    Friday


In [ ]:
# Task 5: total and average per household
print("\nTask 5")
print("Total water used:", df["Water_Usage_Liters"].sum())
household = df.groupby("Household_ID")["Water_Usage_Liters"].agg(["sum", "mean"])
print(household.head())


Task 5
Total water used: 7762823.600000001
                  sum         mean
Household_ID                      
HH00001        1594.4  1594.400000
HH00003        9639.6  9639.600000
HH00004       21835.3  7278.433333
HH00005       27751.5  9250.500000
HH00006       14005.5  7002.750000


In [ ]:
# Task 6: month to month change for each household
print("\nTask 6")
monthly = df.groupby(["Household_ID", "Year_Month"])["Water_Usage_Liters"].sum().reset_index()
monthly["Change"] = monthly.groupby("Household_ID")["Water_Usage_Liters"].diff()
print(monthly.dropna().head(10))


Task 6
   Household_ID Year_Month  Water_Usage_Liters  Change
3       HH00004    2025-10              8305.1  6038.2
4       HH00004    2026-08             11263.3  2958.2
6       HH00005    2026-09             10890.6  -898.8
7       HH00005    2026-10              5071.5 -5819.1
9       HH00006    2025-10              3817.4 -6370.7
11      HH00007    2026-10              2466.3  -926.6
14      HH00009    2026-08              4827.1 -5201.2
16      HH00010    2026-05             10598.7  4421.5
18      HH00011    2025-07              4184.6 -2379.0
19      HH00011    2025-09              4531.3   346.7


In [ ]:
# Task 7: rank household types by total usage
print("\nTask 7")
type_total = df.groupby("Household_Type")["Water_Usage_Liters"].sum().sort_values(ascending=False)
print(type_total)



Task 7
Household_Type
Hostel         2018727.2
Villa          1986901.2
Apartment      1892405.1
Independent    1864790.1
Name: Water_Usage_Liters, dtype: float64


In [ ]:
# Task 8: average usage by building type and zone
print("\nTask 8")
print(df.groupby("Building_Type")["Water_Usage_Liters"].mean())
print(df.groupby("Zone")["Water_Usage_Liters"].mean())



Task 8
Building_Type
Commercial     6444.984533
Mixed Use      6159.742067
Residential    6227.073154
Name: Water_Usage_Liters, dtype: float64
Zone
Zone A    6249.233750
Zone B    6661.986307
Zone C    6028.834457
Zone D    6304.635865
Zone E    6140.598024
Name: Water_Usage_Liters, dtype: float64


In [ ]:
# Task 9: top 10 households
print("\nTask 9")
print(household.sort_values("sum", ascending=False).head(10))



Task 9
                  sum         mean
Household_ID                      
HH00195       71023.9  7891.544444
HH00049       67565.7  7507.300000
HH00364       54217.7  6024.188889
HH00041       50139.6  7162.800000
HH00223       48674.0  6953.428571
HH00198       48039.9  6862.842857
HH00246       45905.9  7650.983333
HH00196       45397.5  7566.250000
HH00066       45057.3  7509.550000
HH00449       44297.5  8859.500000


In [ ]:
# Task 10: month with highest usage
print("\nTask 10")
month_total = df.groupby("Year_Month")["Water_Usage_Liters"].sum()
print("Highest month:", month_total.idxmax(), month_total.max())



Task 10
Highest month: 2025-04 485532.5


In [ ]:
# Task 11: dictionary of zone totals and set of unique zones
print("\nTask 11")
zone_totals = {}
for i in range(len(df)):
    zone = df.iloc[i]["Zone"]
    usage = df.iloc[i]["Water_Usage_Liters"]
    if zone in zone_totals:
        zone_totals[zone] += usage
    else:
        zone_totals[zone] = usage
print(zone_totals)
print(set(df["Zone"]))


Task 11
{'Zone A': np.float64(1499816.1000000003), 'Zone C': np.float64(1609698.8000000007), 'Zone E': np.float64(1553571.300000001), 'Zone B': np.float64(1605538.6999999995), 'Zone D': np.float64(1494198.6999999995)}
{'Zone B', 'Zone D', 'Zone A', 'Zone C', 'Zone E'}


In [ ]:
# Task 12: function to classify usage
# Low = below 25th percentile, High = above 75th percentile, rest Normal
print("\nTask 12")
low_limit = np.percentile(df["Water_Usage_Liters"], 25)
high_limit = np.percentile(df["Water_Usage_Liters"], 75)

def classify(usage):
    if usage < low_limit:
        return "Low"
    elif usage > high_limit:
        return "High"
    else:
        return "Normal"

df["Category"] = df["Water_Usage_Liters"].apply(classify)
print(low_limit, high_limit)
print(df["Category"].value_counts())



Task 12
3612.15 8958.825
Category
Normal    618
Low       310
High      310
Name: count, dtype: int64


In [ ]:
# Task 13: statistics with numpy
print("\nTask 13")
values = df["Water_Usage_Liters"].values
print("Mean:", np.mean(values))
print("Median:", np.median(values))
print("Variance:", np.var(values))
print("Std dev:", np.std(values))



Task 13
Mean: 6270.4552504038775
Median: 6292.05
Variance: 10214321.16113156
Std dev: 3195.985162845967


In [ ]:
# Task 14: outliers using IQR
print("\nTask 14")
q1 = np.percentile(values, 25)
q3 = np.percentile(values, 75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
outliers = df[df["Water_Usage_Liters"] > upper]
print("Upper limit:", upper)
print("Number of outliers:", len(outliers))
print(df.groupby("Household_Type")["Water_Usage_Liters"].mean())



Task 14
Upper limit: 16978.8375
Number of outliers: 0
Household_Type
Apartment      6287.060133
Hostel         6429.067516
Independent    6174.801656
Villa          6189.723364
Name: Water_Usage_Liters, dtype: float64


In [ ]:
# Task 15: WaterMeter class and recommendations
print("\nTask 15")
class WaterMeter:
    def __init__(self, meter_id, household_type):
        self.meter_id = meter_id
        self.household_type = household_type

    def show(self):
        print("Meter:", self.meter_id, "| Type:", self.household_type)

meter = WaterMeter(df.iloc[0]["Meter_Record_ID"], df.iloc[0]["Household_Type"])
meter.show()

print("""
Recommendations:
1. Focus awareness campaigns on the household type with the highest total usage.
2. Check for leaks in the zone with the highest average usage.
3. Send alerts to households in the 'High' category.
4. Run saving campaigns before the month with the highest usage.
5. Install low-flow taps and smart meters in the building type that uses the most water.
""")



Task 15
Meter: WM000000 | Type: Independent

Recommendations:
1. Focus awareness campaigns on the household type with the highest total usage.
2. Check for leaks in the zone with the highest average usage.
3. Send alerts to households in the 'High' category.
4. Run saving campaigns before the month with the highest usage.
5. Install low-flow taps and smart meters in the building type that uses the most water.

